## Cell 1 — Install dependencies
Run this first, then **Runtime → Restart session → re-run from top**

In [1]:
!pip install -q requests
!pip install -q verifia --upgrade
!pip install -q "pytabkit[models]"

## Cell 2 — Download resources (CSV + domain YAML + docs)

In [2]:
print("Using local data files.")
print("data/ and domains/ directories are pre-populated.")

## Cell 3 — Imports

In [3]:
import os
import gc
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from verifia.models import build_from_model_card
from verifia.verification.results import RulesViolationResult
from verifia.context.data import Dataset
from verifia.verification.verifiers import RuleConsistencyVerifier

## Cell 4 — Constants and data loading

In [4]:
RAND_SEED = 0
MODELS_DIRPATH = "models"
DATA_PATH = "data/concrete_compressive_strength.csv"
DOMAIN_PATH = "domains/concrete_compressive_strength.yaml"

dataframe = pd.read_csv(DATA_PATH)
target_name = "concrete_compressive_strength"
feature_names = [col for col in dataframe.columns if col != target_name]

print("Dataset shape:", dataframe.shape)
print("Features:", feature_names)

Dataset shape: (1030, 9)
Features: ['cement', 'blast_furnace_slag', 'fly_ash', 'water', 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age']


## Cell 5 — NamedColumnWrapper

VerifIA's sklearn wrapper internally calls `.to_numpy()` before every `predict()` call,
stripping column names. pytabkit models require the same named columns seen during `fit()`.
This wrapper intercepts numpy inputs and re-attaches column names transparently.

In [5]:
class NamedColumnWrapper(BaseEstimator, RegressorMixin):
    """Wraps a pytabkit model so it accepts numpy arrays by re-attaching column names.
    Required because VerifIA internally strips column names before calling predict().
    """
    def __init__(self, model, feature_names):
        self.model = model
        self.feature_names = feature_names

    def fit(self, X, y):
        df = pd.DataFrame(X, columns=self.feature_names) if not isinstance(X, pd.DataFrame) else X
        self.model.fit(df, y)
        return self

    def predict(self, X):
      df = pd.DataFrame(X, columns=self.feature_names) if not isinstance(X, pd.DataFrame) else X.copy()
      df = df.astype(float)  # ensure numeric — pytabkit rejects object dtype
      return self.model.predict(df)

    def get_params(self, deep=True):
        return {"model": self.model, "feature_names": self.feature_names}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

print("NamedColumnWrapper defined.")

NamedColumnWrapper defined.


## Cell 6 — Split into train / test using VerifIA Dataset

In [6]:
dataset = Dataset(dataframe, target_name, feature_names)
train_dataset, test_dataset = dataset.split(0.8, RAND_SEED)

X_train, y_train = train_dataset.feature_data(), train_dataset.target_data
X_test,  y_test  = test_dataset.feature_data(),  test_dataset.target_data

print("Train size:", X_train.shape)
print("Test  size:", X_test.shape)

Train size: (824, 8)
Test  size: (206, 8)


## Cell 7 — Train RealMLP_TD_Regressor

Replaces the original TF + HPO pipeline.
- `n_cv=1` skips 5-fold CV ensembling (faster, fine for experiments — use `n_cv=5` for best accuracy)
- `device='cpu'` avoids CUDA issues on Colab free tier
- Wrapped in `NamedColumnWrapper` to fix VerifIA/pytabkit column name compatibility

In [7]:
from pytabkit import RealMLP_TD_Regressor

base_model = RealMLP_TD_Regressor(n_cv=1, device='cpu')
model = NamedColumnWrapper(base_model, feature_names)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse  = np.sqrt(mean_squared_error(y_test, preds))
print(f"RealMLP RMSE: {rmse:.3f} MPa  |  TF baseline: 5.756 MPa")
gc.collect()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=256` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


RealMLP RMSE: 4.702 MPa  |  TF baseline: 5.756 MPa


903

## Cell 8 — Build VerifIA model card and wrap model

In [8]:
os.makedirs(MODELS_DIRPATH, exist_ok=True)

model_wrapper = build_from_model_card({
    "name": "realmlp_compressive",
    "version": "1",
    "type": "regression",
    "description": "RealMLP_TD model predicting compressive strength of concrete",
    "framework": "sklearn",
    "feature_names": feature_names,
    "target_name": target_name,
    "local_dirpath": MODELS_DIRPATH
}).wrap_model(model)

print("Model wrapped successfully:", model_wrapper)

Model wrapped successfully: <verifia.models.sklearn.SKLearnModel object at 0x788ac6171e50>


## Cell 9 — Evaluate predictive performance via VerifIA

In [9]:
metric_name, metric_score = model_wrapper.calculate_predictive_performance(test_dataset)
print(f"Test Performance Metric: {metric_name}={metric_score:.4f}")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Test Performance Metric: RMSE=4.7023


## Cell 10 — Prepare test DataFrame
VerifIA's `.on()` method expects a DataFrame with both features AND the target column.

In [10]:
# Build full test DataFrame (features + target) — required by VerifIA
if not isinstance(X_test, pd.DataFrame):
    test_features_df = pd.DataFrame(X_test, columns=feature_names)
else:
    test_features_df = X_test.copy()

# Ensure all feature columns are float — pytabkit requires numeric types
test_features_df = test_features_df.astype(float)

full_test_df = test_features_df.copy()
# Reset index to avoid index misalignment when adding target column
y_vals = y_test.values if hasattr(y_test, 'values') else np.array(y_test)
full_test_df[target_name] = y_vals.astype(float)

print(full_test_df.dtypes)  # All should be float64
print(full_test_df.head())
print("Shape:", full_test_df.shape)

cement                           float64
blast_furnace_slag               float64
fly_ash                          float64
water                            float64
superplasticizer                 float64
coarse_aggregate                 float64
fine_aggregate                   float64
age                              float64
concrete_compressive_strength    float64
dtype: object
     cement  blast_furnace_slag  fly_ash  water  superplasticizer  \
747   500.0                 0.0      0.0  200.0               0.0   
718   122.6               183.9      0.0  203.5               0.0   
175   362.6               189.0      0.0  164.9              11.6   
828   522.0                 0.0      0.0  146.0               0.0   
713   157.0               236.0      0.0  192.0               0.0   

     coarse_aggregate  fine_aggregate   age  concrete_compressive_strength  
747            1125.0           613.0   3.0                          26.06  
718             958.2           800.1   7.0     

## Cell 11 — Domain Configuration (Option A — Predefined YAML)

Uses the same 23-rule domain YAML from the original notebook.
This is the recommended path for reproducible comparison with the TF baseline.

In [11]:
model_verifier = RuleConsistencyVerifier(DOMAIN_PATH)
print("Domain loaded from:", DOMAIN_PATH)

Domain loaded from: ../domains/concrete_compressive_strength.yaml


## Cell 12 — Run VerifIA Verification (PSO)

Using PSO with pop_size=50, max_iters=20 — same search budget as Day 2 TF baseline for fair comparison.

TF baseline to beat:
- Overall inconsistent inputs: 19.85%
- Rules violated: 19 / 23
- Worst rule (R_3): 16.67% consistent

In [ ]:
from tqdm.notebook import tqdm

import logging
import os

os.makedirs("reports", exist_ok=True)

verifier = RuleConsistencyVerifier(DOMAIN_PATH)
n_rules = len(verifier.domain.rules)
print(f"Starting verification: {n_rules} rules, RS pop_size=30, max_iters=15")
print(f"Estimated time: ~{n_rules * 0.5:.0f} min on local CPU")

with tqdm(total=n_rules, desc="Verifying rules", unit="rule") as pbar:
    class TqdmHandler(logging.Handler):
        def emit(self, record):
            msg = self.format(record)
            if "Detected Rule R_" in msg:
                pbar.update(1)
                pbar.set_postfix_str(msg.split("Detected ")[-1][:40])

    handler = TqdmHandler()
    logging.getLogger("verifia.verification.verifiers").addHandler(handler)

    result: RulesViolationResult = (
        verifier
        .verify(model_wrapper)
        .on(full_test_df)
        .using("RS")
        .run(pop_size=30, max_iters=15)
    )

    logging.getLogger("verifia.verification.verifiers").removeHandler(handler)

result.save_as_html("reports/realmlp_verifia_report.html")
print("Done! Report saved to reports/realmlp_verifia_report.html")

## Cell 13 — Print summary

In [ ]:
print("=== VerifIA Verification Summary ===")
print(result)

## Cell 14 — Save model and model card

In [ ]:
model_wrapper.save_model()
model_wrapper.save_model_card(f"{MODELS_DIRPATH}/realmlp_compressive.yaml")
print("Model and model card saved.")